### Business Problem: the lack of clear data governance and process mapping
1. Similarity Score: Calculate similarity between columns/tables.
2. Process Mapping: Map table connections via keys and flows to trace data lineage across systems.

### Package Import

In [1]:
# ! pip install snowflake

In [ ]:
# Standard Library
import os
import re
import ast
import random
import tempfile
import warnings
from pathlib import Path
from datetime import datetime
from itertools import combinations
from collections import Counter, defaultdict
import textwrap

# Data Handling & Stats
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, entropy

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

# Progress Display
from tqdm import tqdm

# Cryptography
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.fernet import Fernet

# Office365 / SharePoint
from office365.runtime.auth.user_credential import UserCredential
from office365.sharepoint.client_context import ClientContext

# Snowflake
import snowflake.connector as sf
from snowflake.connector.pandas_tools import write_pandas

# SQLAlchemy
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

# Graph / Network
import networkx as nx

# Pandas Display & Warnings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')


# 1. Preprocessing - Download data from sharepoint folder

In [ ]:
site_url = "https://tintl.sharepoint.com/sites/XXXX"#sharepoint site url
folder_relative_url ="folder url" #the sharepoint folder stores data
save_path = "../data/events/"

def download_folder_contents_from_sharepoint():
    ctx = ClientContext(site_url).with_credentials(UserCredential(user, pw))
    folder = ctx.web.get_folder_by_server_relative_url(folder_relative_url).expand(["Files"]).get().execute_query()
    
    print(folder.files)
    
    # 2. start download process (per file)
    for file in folder.files:
        download_file_name = os.path.join(save_path, os.path.basename(file.properties["Name"]))
        with open(download_file_name, "wb") as local_file:
            file.download(local_file)
            ctx.execute_query()
        print("[Ok] file has been downloaded: {0}".format(download_file_name))

##### Usage Function Below
download_folder_contents_from_sharepoint()

# 2. Filtered by Context and Run Similarity (computes JS, KS)

In [ ]:
def compute_column_similarities(folder_path, output_path="../script/similarity_output", use_tqdm=False):
    os.makedirs(output_path, exist_ok=True)
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    dataframes = {
        file: pd.read_csv(os.path.join(folder_path, file))
        for file in csv_files
    }

    file_pairs = list(combinations(dataframes.keys(), 2))
    file_iter = tqdm(file_pairs, desc="Comparing file pairs") if use_tqdm else file_pairs

    for file_a, file_b in file_iter:
        print(f"\nComparing: {file_a} ↔ {file_b}")
        df_a = dataframes[file_a]
        df_b = dataframes[file_b]

        results = []

        for col_a in df_a.columns:
            for col_b in df_b.columns:
                #print(f"   Column Pair: {col_a} ↔ {col_b}")
                series_a = df_a[col_a].dropna().astype(str).sort_values()
                series_b = df_b[col_b].dropna().astype(str).sort_values()

                set_a = set(series_a)
                set_b = set(series_b)
                intersection = set_a.intersection(set_b)
                union = set_a.union(set_b)

                pa_js = len(intersection) / len(union) if union else 0
                b_in_a = len(intersection) / len(set_a) if set_a else 0
                a_in_b = len(intersection) / len(set_b) if set_b else 0

                try:
                    num_a = pd.to_numeric(series_a, errors='coerce').dropna()
                    num_b = pd.to_numeric(series_b, errors='coerce').dropna()
                    ks_stat, ks_p = ks_2samp(num_a, num_b)
                except Exception:
                    ks_stat, ks_p = np.nan, np.nan

                unique_vals_a = len(set_a)
                unique_vals_b = len(set_b)
                ratio_a = unique_vals_a / len(df_a) if len(df_a) else 0
                ratio_b = unique_vals_b / len(df_b) if len(df_b) else 0

                avg_len_a = series_a.str.len().mean()
                avg_len_b = series_b.str.len().mean()
                card_diff = abs(avg_len_a - avg_len_b)
                card_max = max(avg_len_a, avg_len_b)
                norm_card_diff = card_diff / card_max if card_max else 0

                result = {
                    "File A": file_a,
                    "File B": file_b,
                    "Column A": col_a,
                    "Column B": col_b,
                    "paJS": pa_js,
                    "B_intersection_A": b_in_a,
                    "A_intersection_B": a_in_b,
                    "KS Statistic": ks_stat,
                    "KS P-value": ks_p,
                    "Table_A Rows": len(df_a),
                    "Table_B Rows": len(df_b),
                    "Column A Unique Vals": unique_vals_a,
                    "Column B Unique Vals": unique_vals_b,
                    "Table A Unique Ratio": ratio_a,
                    "Table B Unique Ratio": ratio_b,
                    "Column A Cardinality": avg_len_a,
                    "Column B Cardinality": avg_len_b,
                    "Cardinality Difference": norm_card_diff
                }

                results.append(result)

        # Save to CSV for each table pair
        base_name_a = os.path.splitext(file_a)[0]
        base_name_b = os.path.splitext(file_b)[0]
        out_filename = f"{base_name_a}__vs__{base_name_b}_similarity.csv"
        out_path = os.path.join(output_path, out_filename)

        pd.DataFrame(results).to_csv(out_path, index=False)
        print(f"Saved similarity results to {out_path}")

    print("\nAll comparisons completed.")


In [ ]:
# find candidates for contexts
folder_path = Path("../data/events")  
csv_files = list(folder_path.glob("*.csv"))  # all CSV files in folder

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    print(f"\nFile: {csv_file.name} (Rows: {len(df)})")
    
    for col in df.columns:
        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]) or pd.api.types.is_timedelta64_dtype(df[col]) or col.endswith("IK"):
            continue
        
        unique_vals = df[col].nunique(dropna=True)
        completeness = df[col].notna().mean()  # fraction of non-null values

        # Only keep columns with unique values between 2 and 4 (inclusive lower, exclusive upper bound 5)
        if 2 <= unique_vals < 5 and completeness >= 0.8:
            print(f"  Column: {col} | Unique values: {unique_vals} | Completeness: {completeness:.2%}")

In [ ]:
def load_csvs_with_context(folder_path, context_filters, max_rows=None):
    folder_path = Path(folder_path)
    csv_files = list(folder_path.glob("*.csv"))# all csv files in the folder
    dfs = {}

    for csv_file in csv_files:
        df = pd.read_csv(csv_file, nrows=max_rows)
        filtered_df = df.copy()

        # Apply only the filters where the column exists in this CSV
        for col, filter_val in context_filters.items():
            if col in filtered_df.columns:
                if callable(filter_val):
                    filtered_df = filtered_df[filtered_df[col].apply(filter_val)]
                else:
                    filtered_df = filtered_df[filtered_df[col] == filter_val]

        # Drop columns with only 1 unique value or completeness < 20%
        cols_to_drop = []
        for col in filtered_df.columns:
            completeness = filtered_df[col].notna().sum() / len(filtered_df)
            if filtered_df[col].nunique() <= 1 or completeness < 0.2:
                cols_to_drop.append(col)

        filtered_df = filtered_df.drop(columns=cols_to_drop)

        dfs[csv_file.name] = filtered_df

    return dfs


## context filtering

In [ ]:
contexts = {
    "TYPEA": lambda x: x in [2],
    "TYPEB": lambda x: x in [0]
}
folder = "../data/events"

In [ ]:
dfs = load_csvs_with_context(folder, contexts, max_rows=10000)# random 10,000 rows for each table

# Check results
for fname, df_filtered in dfs.items():
    print(f"{fname}: {len(df_filtered)} rows after filtering")

In [ ]:
# number fo columns for each table
for fname, df_filtered in dfs.items():
    print(f"{fname}: {len(df_filtered.columns)} columns")

In [ ]:
def compute_column_similarities_from_context(
    context_dfs, 
    output_path="../script/similarity_output-sample", 
    use_tqdm=False
):

    os.makedirs(output_path, exist_ok=True)
    file_pairs = list(combinations(context_dfs.keys(), 2))
    file_iter = tqdm(file_pairs, desc="Comparing file pairs") if use_tqdm else file_pairs

    for file_a, file_b in file_iter:
        out_filename = f"{os.path.splitext(file_a)[0]}__vs__{os.path.splitext(file_b)[0]}_similarity.csv"
        out_path = os.path.join(output_path, out_filename)

        print(f"\nComparing: {file_a} ↔ {file_b}")
        df_a = context_dfs[file_a]
        df_b = context_dfs[file_b]

        results = []

        for col_a in df_a.columns:
            for col_b in df_b.columns:
                series_a = df_a[col_a].dropna().astype(str).sort_values()
                series_b = df_b[col_b].dropna().astype(str).sort_values()

                set_a = set(series_a)
                set_b = set(series_b)
                intersection = set_a.intersection(set_b)
                union = set_a.union(set_b)

                pa_js = len(intersection) / len(union) if union else 0
                b_in_a = len(intersection) / len(set_a) if set_a else 0
                a_in_b = len(intersection) / len(set_b) if set_b else 0

                try:
                    num_a = pd.to_numeric(series_a, errors='coerce').dropna()
                    num_b = pd.to_numeric(series_b, errors='coerce').dropna()
                    ks_stat, ks_p = ks_2samp(num_a, num_b)
                except Exception:
                    ks_stat, ks_p = np.nan, np.nan

                results.append({
                    "File A": file_a,
                    "File B": file_b,
                    "Column A": col_a,
                    "Column B": col_b,
                    "paJS": pa_js,
                    "B_intersection_A": b_in_a,
                    "A_intersection_B": a_in_b,
                    "KS Statistic": ks_stat,
                    "KS P-value": ks_p,
                    "Table_A Rows": len(df_a),
                    "Table_B Rows": len(df_b),
                    "Column A Unique Vals": len(set_a),
                    "Column B Unique Vals": len(set_b),
                    "Table A Unique Ratio": len(set_a) / len(df_a) if len(df_a) else 0,
                    "Table B Unique Ratio": len(set_b) / len(df_b) if len(df_b) else 0,
                    "Column A Cardinality": series_a.str.len().mean(),
                    "Column B Cardinality": series_b.str.len().mean(),
                    "Cardinality Difference": abs(series_a.str.len().mean() - series_b.str.len().mean()) / max(series_a.str.len().mean(), series_b.str.len().mean()) if max(series_a.str.len().mean(), series_b.str.len().mean()) else 0
                })

        pd.DataFrame(results).to_csv(out_path, index=False)
        print(f"Saved similarity results to {out_path}")

    print("\nAll comparisons completed.")


In [ ]:
dfs.keys()

# 3. Find PK Prep

## Similarity Matrix

In [ ]:
# merge all similarity tables into one df
def merge_csv_files(folder_path):
    # List all CSV files in the folder
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    # Read and concatenate all CSVs into one DataFrame
    merged_df = pd.concat(
        [pd.read_csv(os.path.join(folder_path, file)) for file in csv_files],
        ignore_index=True
    )
    
    return merged_df

In [ ]:
df_merged = merge_csv_files("similarity_output")

In [ ]:
len(df_merged)

## similarity score



In [ ]:
# count how many different files each column name appears in 
df2 = df_merged
pkcandrows2 = []

for _, row in df2.iterrows():
    pkcandrows2.append({"File": row["File A"], "Column": row["Column A"]})
    pkcandrows2.append({"File": row["File B"], "Column": row["Column B"]})

pkcand_df2 = pd.DataFrame(pkcandrows2)
pkcand_df2.drop_duplicates(inplace = True)
pkcand_df2.reset_index(drop=True, inplace = True)

file_counts2 = pkcand_df2.groupby("Column")["File"].nunique().reset_index()
file_counts2.rename(columns={"File": "File Count"}, inplace=True)

# merge back to add 'File Count' column
pkcand_df_final2 = pkcand_df2.merge(file_counts2, on="Column")
pkcand_df_final2 = pkcand_df_final2[pkcand_df_final2["File Count"]>2].sort_values(by="File Count", ascending=False).reset_index(drop=True)#at least exist in 2 excels

pkcand_df_final2.head(10)

In [ ]:
len(pkcand_df_final2.Column.unique())

In [ ]:
pkcand_df_final2

In [ ]:
# remove categorical - numeric pairs

threshold = 5  # categorical vs numerical cutoff, filter out numeric and categorical pair

df2_filtered = df2[
    ((df2['Column A Unique Vals'] <= threshold) & (df2['Column B Unique Vals'] <= threshold)) |
    ((df2['Column A Unique Vals'] > threshold) & (df2['Column B Unique Vals'] > threshold))
]

In [ ]:
def compute_similarity(row):
    # Extract values
    paJS = row['paJS']
    b_inter = row['B_intersection_A']
    a_inter = row['A_intersection_B']
    ks_stat = row['KS Statistic']

    # Numerical if BOTH columns > threshold
    is_numerical = (row['Column A Unique Vals'] > threshold) and (row['Column B Unique Vals'] > threshold)
    
    if is_numerical:
        # Numerical similarity
        return max(paJS, b_inter, a_inter, 1 - ks_stat)
    else:
        # Categorical similarity
        return max(paJS, b_inter, a_inter)


In [ ]:
# Create Similarity column
df2_filtered['Similarity'] = df2_filtered.apply(compute_similarity, axis=1)

In [ ]:
df2_final = df2_filtered[df2_filtered['Similarity'] >= 0.8]#set similarity to 0.8

## similarity threshold as 0.8, if choose Oa or Ob, combine with KS p-value > 0.05

In [ ]:
df2_filtered_nonzero = df2_final[
    (df2_final["Similarity"] == df2_final["paJS"]) |
    (
        (df2_final["Similarity"] == df2_final["B_intersection_A"]) &
        (df2_final["KS P-value"] > 0.05)
    ) |
    (
        (df2_final["Similarity"] == df2_final["A_intersection_B"]) &
        (df2_final["KS P-value"] > 0.05)
    )
].copy()

In [ ]:
df2_filtered_nonzero.reset_index(inplace = True, drop = True)

In [ ]:
df_sim = df2_filtered_nonzero

# Get unique tables
tables = sorted(set(df_sim['File A']).union(df_sim['File B']))

# Initialize matrix
sim_matrix = pd.DataFrame(index=tables, columns=tables, data=0.0)

# Fill in maximum similarity for each table pair
for _, row in df_sim.iterrows():
    file_a = row['File A']
    file_b = row['File B']
    sim = row['Similarity']

    sim_matrix.loc[file_a, file_b] = max(sim_matrix.loc[file_a, file_b], sim)
    sim_matrix.loc[file_b, file_a] = max(sim_matrix.loc[file_b, file_a], sim)  

# Fill diagonal with 1.0
for t in tables:
    sim_matrix.loc[t, t] = 1.0

sim_matrix_str = sim_matrix.applymap(lambda x: f"{x:.4f}")

In [ ]:
test = pd.read_csv("../data/events/SCD_TRANSACTION.SCDAT.TRANSMAIN.csv")

In [ ]:
test.columns

'CRETS' in test.columns #TRUE

'CRETS' in test2.columns #TRUE

In [ ]:
test2 =  pd.read_csv("../data/events/SCD_TRANSACTION.SCDAT.ORDERMAIN.csv")

In [ ]:
test2.columns

In [ ]:
test3 =  pd.read_csv("../data/events/SCD_TRANSACTION.SCDAT.TRANSORDER.csv")

In [ ]:
test4 = pd.read_csv("../data/events/SCD_TRANSACTION.SCDAT.TRANSSETTLE.csv")

'CRETS' in test3.columns #FALSE

'CRETS' in test4.columns #FALSE

In [ ]:
len(test2.RECSEQNO.unique().tolist())

In [ ]:
len(test.RECSEQNO.unique().tolist())

In [ ]:
def find_pk_candidates_in_folder(folder, max_rows=10000):
    all_candidates = []
    col_to_files = defaultdict(list)
    
    # 1. Scan all CSVs and compute candidates
    for fname in os.listdir(folder):
        if not fname.endswith(".csv"):
            continue
        fpath = os.path.join(folder, fname)
        
        try:
            df = pd.read_csv(fpath, nrows=max_rows)
        except Exception as e:
            print(f"Skipping {fname} (error: {e})")
            continue
        
        n_rows = len(df)
        if n_rows == 0:
            continue
        
        for col in df.columns:
            series = df[col]
            
            # must be datetime-like
            if not is_datetime_column(series):
                continue
            
            non_null_count = series.notna().sum()
            completeness = non_null_count / n_rows if n_rows > 0 else 0
            
            unique_count = series.nunique(dropna=True)
            uniqueness = unique_count / non_null_count if non_null_count > 0 else 0
            
            if uniqueness > 0.8 and completeness > 0.95:
                all_candidates.append({
                    "File": fname,
                    "Column": col,
                    "Uniqueness": round(uniqueness, 3),
                    "Completeness": round(completeness, 3)
                })
                col_to_files[col].append(fname)
    
    # 2. Add File Count column
    for cand in all_candidates:
        cand["File Count"] = len(col_to_files[cand["Column"]])
    
    return pd.DataFrame(all_candidates)


# 4. find datetime columns

## set threshold

In [ ]:
# Define thresholds
Nunique_min = 10  # Example threshold for unique values
sigma_min = 10  # Example threshold for standard deviation
IQR_min = 500  # Example threshold for IQR
H_min = 0.5  # Example threshold for entropy
time_sigma_min = 100  # Example threshold for time variation (in seconds or smaller unit)


In [ ]:
# Define a function to check if the column has a datetime pattern
def likely_datetime_format(series):
    datetime_pattern = r'\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2}'
    sample = series.dropna().astype(str).head(10)
    match_count = sum(sample.str.contains(datetime_pattern, regex=True))
    return match_count > 0  # Return True if most of the sample looks like datetime

# Function to check if a column can be converted to datetime reliably
def is_datetime_column(series):
    # Check if the column is an object or already datetime
    if series.dtype == 'object' or likely_datetime_format(series):
        try:
            pd.to_datetime(series, errors='raise')  # Try to convert to datetime
            return True
        except (ValueError, TypeError):
            return False
    return False

# Function to compute metrics for datetime columns
def compute_datetime_metrics(df):
    valid_columns = []
    
    for col in df.columns:
        if is_datetime_column(df[col]):  # Check if the column can be treated as datetime or date
            # Convert column to datetime format, then to UNIX timestamp for computations
            
            print('Converting to datetime format...')
            
            df[col] = pd.to_numeric(pd.to_datetime(df[col], errors='coerce'))
            
            # Calculate the number of unique values
            Nunique = df[col].nunique()
            
            # Calculate standard deviation
            sigma = df[col].std()
            
            # Calculate IQR
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            # Calculate entropy
            value_counts = df[col].value_counts(normalize=True)
            H = entropy(value_counts, base=2)
            
            # Extract the time component to check timestamp variation
            original_col = pd.to_datetime(df[col], errors='coerce')
            time_component = original_col.dt.hour * 3600 + original_col.dt.minute * 60 + original_col.dt.second  # Convert time to seconds
            
            time_sigma = time_component.std()  # Calculate time standard deviation
            # Apply conditions
            C1 = 1 if Nunique >= Nunique_min else 0
            C2 = 1 if sigma >= sigma_min else 0
            C3 = 1 if IQR >= IQR_min else 0
            C4 = 1 if H >= H_min else 0
            C5 = 1 if time_sigma >= time_sigma_min else 0  # Check time variation
            
            # Calculate total criteria met
            total_criteria_met = C1 + C2 + C3 + C4 + C5
            
            # If column meets at least 2 criteria, include it
            if total_criteria_met >= 2:
                valid_columns.append(col)
    
    return valid_columns



def compute_best_datetime_columns(df):
    metrics = []

    for col in df.columns:
        if is_datetime_column(df[col]):  
            total = len(df[col]) 
            col_datetime = pd.to_datetime(df[col], errors='coerce')
            col_numeric = pd.to_numeric(col_datetime)

            non_missing_count = col_datetime.notna().sum()
            completeness = non_missing_count / total if total > 0 else 0

            if completeness < 1:  # skip if less than 95% complete
                continue

            Nunique = col_numeric.nunique()
            sigma = col_numeric.std()
            Q1, Q3 = col_numeric.quantile([0.25, 0.75])
            IQR = Q3 - Q1
            value_counts = col_numeric.value_counts(normalize=True)
            H = entropy(value_counts, base=2)

            time_component = (
                col_datetime.dt.hour * 3600
                + col_datetime.dt.minute * 60
                + col_datetime.dt.second
            )
            time_sigma = time_component.std()

            metrics.append({
                "col": col,
                "Nunique": Nunique,
                "sigma": sigma,
                "IQR": IQR,
                "H": H,
                "time_sigma": time_sigma,
                "completeness": completeness
            })

    if not metrics:
        return []

    # compute thresholds (75th percentile)
    thresholds = {}
    for key in ["Nunique", "sigma", "IQR", "H", "time_sigma", "completeness"]:
        values = [m[key] for m in metrics if pd.notna(m[key])]
        thresholds[key] = np.percentile(values, 75) if values else float("inf")

    # score assignment
    for m in metrics:
        m["score"] = sum(
            1 for key in thresholds
            if pd.notna(m[key]) and m[key] >= thresholds[key]
        )

    # pick best col(s)
    best_col = max(metrics, key=lambda x: (x["score"], x["sigma"], x["completeness"]))
    return [best_col["col"]]

In [ ]:
# Iterate through folder of Excel files
folder_path = f'../data/events'
# Initialize a list to store the results for each file
results1 = []

for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path)
        
        # Compute valid datetime or date columns
        valid_columns = compute_datetime_metrics(df)
        
        # Store the file name and the corresponding valid columns
        results1.append({'FILE_NAME': file_name, 'DATETIME_COL_SELECTED': ', '.join(valid_columns)})

# Convert results to a dataframe
results_df1 = pd.DataFrame(results1)

In [ ]:
results_df1

## choose the best datetime column for each table

In [ ]:
# Iterate through folder of Excel files
folder_path = f'../data/events'
# Initialize a list to store the results for each file
results = []

for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path)
        
        # Compute valid datetime or date columns
        valid_columns = compute_best_datetime_columns(df)
        
        # Store the file name and the corresponding valid columns
        results.append({'FILE_NAME': file_name, 'PICKED_DATETIME_COL': ', '.join(valid_columns)})

# Convert results to a dataframe
results_df = pd.DataFrame(results)

In [ ]:
# Condition for empty or missing datetime columns
is_empty = results_df['PICKED_DATETIME_COL'].isna() | (results_df['PICKED_DATETIME_COL'].str.strip() == '')

# DataFrame with empty or missing datetime columns
reference_tables = results_df.loc[is_empty, ['FILE_NAME', 'PICKED_DATETIME_COL']]

# DataFrame with non-empty datetime columns
event_tables = results_df.loc[~is_empty, ['FILE_NAME', 'PICKED_DATETIME_COL']]

In [ ]:
event_tables

### one datetime column for each event table- result_df

In [ ]:
event_tables_non_picked = event_tables[event_tables['PICKED_DATETIME_COL'].isna()].copy()
event_tables_picked = event_tables[event_tables['PICKED_DATETIME_COL'].notna()].copy()

In [ ]:
event_tables_picked

# 5. find primary keys

In [ ]:
def find_primary_key_candidates(df, exclude_cols=None):
    """
    Find candidate primary key columns in a DataFrame, excluding datetime-like columns.
    """
    results = []
    exclude_cols = set(exclude_cols) if exclude_cols else set()

    for col in df.columns:
        if col in exclude_cols:
            continue

        series = df[col].dropna()
        if len(series) == 0:
            continue

        completeness = len(series) / len(df)
        uniqueness = series.nunique() / len(df)

        # Candidate PK: high uniqueness + high completeness
        if completeness > 0.9 and uniqueness > 0.9:  
            results.append({
                "Column": col,
                "Completeness": round(completeness, 3),
                "Uniqueness": round(uniqueness, 3)
            })

    return results



folder_path = f'../data/events'
results = []
pk_results = []

for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path)

        # Step 1: find datetime-like columns
        valid_columns = compute_datetime_metrics(df)
        results.append({
            'FILE_NAME': file_name,
            'DATETIME_COL_SELECTED': ', '.join(valid_columns)
        })

        # Step 2: find PK candidates, excluding those datetime cols
        pk_candidates = find_primary_key_candidates(df, exclude_cols=valid_columns)
        for cand in pk_candidates:
            pk_results.append({
                'FILE_NAME': file_name,
                'Column': cand["Column"],
                'Completeness': cand["Completeness"],
                'Uniqueness': cand["Uniqueness"]
            })

# DataFrames
results_df = pd.DataFrame(results)         # datetime columns
pk_results_df = pd.DataFrame(pk_results)   # primary key candidates


In [ ]:
# All unique PK candidates
unique_columns = set(pkcand_df_final2['Column'].unique())

# Collect all datetime columns across results_df1
all_datetime_cols = set()
for cols in results_df1['DATETIME_COL_SELECTED'].dropna():
    for col in cols.split(','):
        all_datetime_cols.add(col.strip())

#  Filter: PK candidates that are NOT datetime columns
pk_candidates = unique_columns - all_datetime_cols

print("Primary key candidates (excluding datetime columns):")
print(pk_candidates)


# 6. random sampling

In [ ]:
folder_path = '../data/events'

# Example primary key list
p_key = list(pk_candidates)  # p_key list may change later, after discussion

# event_tables: FILE_NAME -> PICKED_DATETIME_COL mapping
event_tables_dict = dict(zip(event_tables['FILE_NAME'], event_tables['PICKED_DATETIME_COL']))

# Load all CSVs into memory for faster matching
all_tables = {}
for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        all_tables[file_name] = pd.read_csv(file_path)

In [ ]:
output = []

for pk in p_key:
    # Find tables that contain this primary key
    tables_with_pk = [fname for fname, df in all_tables.items() if pk in df.columns]
    
    for table_name in tables_with_pk:
        df = all_tables[table_name]
        sample_ids = random.sample(df[pk].dropna().unique().tolist(), min(2000, df[pk].nunique()))
        
        for sid in sample_ids:
            sequence = []
            
            # Check all tables for this id
            for t_name, t_df in all_tables.items():
                if pk in t_df.columns and sid in t_df[pk].values:
                    dt_col = event_tables_dict.get(t_name, None)
                    if dt_col and dt_col in t_df.columns:
                        dt_values = t_df.loc[t_df[pk] == sid, dt_col].tolist()
                        for dt_val in dt_values:
                            sequence.append((t_name, dt_col, dt_val))
            
            output.append({
                'primary_key': pk,
                'id': sid,
                'sequence': sequence
            })
output_df = pd.DataFrame(output)

In [ ]:
output_df.head()

In [ ]:
filtered_output = output_df[output_df['sequence'].apply(lambda x: len(x) >= 2)].copy()# at least has 3 tables

In [ ]:
len(filtered_output)

In [ ]:
filtered_output.primary_key.value_counts()

filtered_output.head()

In [ ]:
output_df.to_csv('../script/filtered_output_not_hardcoded.csv', index=False)

In [ ]:
def remove_consecutive_duplicates(seq):
    if not seq:
        return []
    new_seq = [seq[0]]
    for t in seq[1:]:
        if t != new_seq[-1]:
            new_seq.append(t)
    return new_seq

# Build sequence_tables with tuple sequences
sequence_tables = pd.DataFrame({
    "Primary_Key": filtered_output["primary_key"],
    "id": filtered_output["id"],
    "sequence": filtered_output["sequence"].apply(
        lambda seq: tuple(
            remove_consecutive_duplicates(
                [t[0] for t in sorted(seq, key=lambda x: x[2] if x[2] is not None else pd.Timestamp.max)]
            )
        )
    )
})

# Split cleaned sequence into table pairs
sequence_tables['sequence_pairs'] = sequence_tables['sequence'].apply(
    lambda seq: [(seq[i], seq[i + 1]) for i in range(len(seq) - 1)] if len(seq) > 1 else []
)

sequence_tables.reset_index(drop=True, inplace=True)
sequence_tables = sequence_tables[sequence_tables['sequence'].apply(lambda x: len(x) > 1)].reset_index(drop=True)

In [ ]:
sequence_tables.head()

In [ ]:
whole_seq_counts = defaultdict(dict)

# Group by Primary Key
for pk, group in sequence_tables.groupby('Primary_Key'):
    # Convert each sequence list to a tuple (hashable) and collect
    all_seqs = [tuple(seq) for seq in group['sequence']]
    
    # Count occurrences of each exact sequence
    seq_counts = dict(Counter(all_seqs))
    
    whole_seq_counts[pk] = seq_counts

# Example output
for pk, counts in whole_seq_counts.items():
    print(f"Primary Key: {pk}")
    for seq, count in counts.items():
        print(f"Sequence: {seq}, Count: {count}")

In [ ]:
sorted_transik = dict(sorted(whole_seq_counts['TRANSIK'].items(), key=lambda x: x[1], reverse=True))
pk_edge_counts = defaultdict(dict)

# group by pk
for pk, group in sequence_tables.groupby('Primary_Key'):
    # Merge all sequence_pairs lists into one
    all_pairs = [pair for sublist in group['sequence_pairs'] for pair in sublist]
    
    # Count occurrences of each pair
    pair_counts = dict(Counter(all_pairs))
    
    pk_edge_counts[pk] = pair_counts

In [ ]:
pk_edge_counts

In [ ]:

def scale(value, src_min, src_max, dst_min, dst_max):
    if src_max == src_min:
        return (dst_min + dst_max) / 2
    return dst_min + (value - src_min) * (dst_max - dst_min) / (src_max - src_min)

def wrap_label(label, width=15):
    return '\n'.join(textwrap.wrap(label, width))

def compute_edge_counts(sequence_tables, primary_key=None):
    """
    Compute counts of each table pair (edge) for the whole table or per PK
    """
    edge_counter = Counter()
    if primary_key:
        df = sequence_tables[sequence_tables['Primary_Key'] == primary_key]
    else:
        df = sequence_tables

    for pairs in df['sequence_pairs']:
        edge_counter.update(pairs)
    return dict(edge_counter)

# Draw function
def draw_multidigraph_from_edge_counts(edge_counts, title="Graph"):
    # Extract all nodes
    all_nodes = set()
    for src, dst in edge_counts.keys():
        all_nodes.add(src)
        all_nodes.add(dst)
    #build G
    G = nx.MultiDiGraph()
    G.add_nodes_from(all_nodes) 
    
    # Assign colors
    colors = list(mcolors.TABLEAU_COLORS.values())
    random.shuffle(colors)
    color_map = {node: colors[i % len(colors)] for i, node in enumerate(sorted(all_nodes))}


    for (src, dst), w in edge_counts.items():
        G.add_edge(src, dst, weight=w)

    
    # Compute node sizes
    node_weight = {node: sum(data['weight'] for _, _, data in G.in_edges(node, data=True)) +
                            sum(data['weight'] for _, _, data in G.out_edges(node, data=True))
                   for node in G.nodes()}
    weights = list(node_weight.values())
    min_w, max_w = min(weights), max(weights)
    px_min, px_max = 5, 20
    sizes = [3.1415 * (scale(w, min_w, max_w, px_min, px_max)/2)**2 * 10 for w in weights]
    node_colors = [color_map[node] for node in G.nodes()]

    # Plot
    plt.figure(figsize=(6,6))
    ax = plt.gca()
    pos = nx.spring_layout(G, seed=42, k=2)

    # Node drawing
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_colors,
                           edgecolors='black', linewidths=1.5)
    # Build a cleaned label dictionary
    clean_labels = {n: wrap_label(n.split('.')[-2]) for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=clean_labels, font_size=7)
    #nx.draw_networkx_labels(G, pos, labels={n: wrap_label(n) for n in G.nodes()}, font_size=7)

    # Draw edges with weights
    all_weights = [data['weight'] for _, _, data in G.edges(data=True)]
    min_ew, max_ew = min(all_weights), max(all_weights)
    node_size_dict = dict(zip(G.nodes(), sizes))

    for u, v, data in G.edges(data=True):
        w = data['weight']
        width = scale(w, min_ew, max_ew, 1, 5)
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        dx, dy = x2 - x1, y2 - y1
        dist = (dx**2 + dy**2)**0.5
        if dist == 0: 
            continue
        r_source = (node_size_dict[u]/3.1415)**0.5 / 100
        r_target = (node_size_dict[v]/3.1415)**0.5 / 100
        start_x = x1 + dx*r_source/dist
        start_y = y1 + dy*r_source/dist
        end_x = x2 - dx*r_target/dist
        end_y = y2 - dy*r_target/dist

        arrow = mpatches.FancyArrowPatch(
            (start_x, start_y), (end_x, end_y),
            arrowstyle='-|>', mutation_scale=15, linewidth=width, color='gray'
        )
        ax.add_patch(arrow)
        # Edge label
        t = 0.75
        label_x = x1 + dx*t
        label_y = y1 + dy*t
        ax.text(label_x, label_y, str(w), fontsize=7,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
                ha='center', va='center')

    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
sequence_tables.head()

In [ ]:
# Draw graph for each primary key
for pk in sequence_tables['Primary_Key'].unique():
    edge_counts_pk = compute_edge_counts(sequence_tables, primary_key=pk)
    if edge_counts_pk:  # only draw if there are edges
        draw_multidigraph_from_edge_counts(edge_counts_pk, title=pk)


# Appendix

In [ ]:
def analyze_timestamps_with_picked_col(sampled_ids, tables_to_check, event_tables_picked, data_path, p_key):
    result = {}

    for id_ in sampled_ids:
        timestamp_values = []
        timestamp_cols = []

        for table_name in tables_to_check:
            # Get picked datetime col for this table
            picked_col_series = event_tables_picked.loc[event_tables_picked['FILE_NAME'] == table_name, 'PICKED_DATETIME_COL']
            if len(picked_col_series) == 0:
                timestamp_col = None
            else:
                timestamp_col = picked_col_series.values[0]

            timestamp_cols.append((table_name, timestamp_col))

            if timestamp_col is not None:
                try:
                    df = pd.read_csv(f"{data_path}/{table_name}", usecols=[p_key, timestamp_col])
                    ts_vals = df.loc[df[p_key] == id_, timestamp_col].values
                    ts_val = ts_vals[0] if len(ts_vals) > 0 else pd.NaT
                except Exception as e:
                    print(f"Warning: failed to read {table_name} or column: {e}")
                    ts_val = pd.NaT
            else:
                ts_val = pd.NaT

            timestamp_values.append((table_name, ts_val))

        # Sort tables by timestamp ascending, missing dates last
        def sort_key(x):
            try:
                return pd.to_datetime(x[1])
            except Exception:
                return pd.Timestamp.max

        tables_sorted = [tbl for tbl, ts in sorted(timestamp_values, key=sort_key)]

        result[id_] = {
            'timestamp_value': timestamp_values,
            'related_tables_sorted': tables_sorted,
            'timestamp_col_name': timestamp_cols
        }

    return result


In [ ]:
all_results = {}

for candidate_key, sampled_ids in samples.items():
    print(f"Analyzing timestamps for candidate PK: {candidate_key} with {len(sampled_ids)} sampled IDs")

    # Determine related tables to check for this candidate_key:
    # Example: all files in event_tables_picked, or filter those that contain candidate_key?
    tables_to_check = event_tables_picked['FILE_NAME'].tolist()

    # Run your analysis function for each candidate's sampled IDs
    result = analyze_timestamps_with_picked_col(sampled_ids, tables_to_check, event_tables_picked, fpath, candidate_key)

    all_results[candidate_key] = result

In [ ]:
all_results

In [ ]:
def count_rule_frequencies(result):
    # Outer dict: pk candidate -> dict of rule tuple -> frequency count
    rule_freq = {}

    for pk_candidate, id_dict in result.items():
        freq_dict = defaultdict(int)

        for id_, details in id_dict.items():
            # Get the ordered sequence of tables for this id
            related_tables_sorted = details.get('related_tables_sorted', [])

            # Convert to tuple (ordered, because sequence matters)
            rule = tuple(related_tables_sorted)

            freq_dict[rule] += 1

        rule_freq[pk_candidate] = dict(freq_dict)

    return rule_freq

In [ ]:
rule_freq = count_sorting_rules(all_results)
for pk, rules in rule_freq.items():
    print(f"Primary Key Candidate: {pk}")
    for rule, freq in rules.items():
        print(f"  Rule {rule}: {freq} times")

In [ ]:
rule_freq

In [ ]:
! pip install graphviz

In [ ]:
from graphviz import Digraph
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def scale(value, src_min, src_max, dst_min, dst_max):
    if src_max == src_min:
        return (dst_min + dst_max) / 2
    return dst_min + (value - src_min) * (dst_max - dst_min) / (src_max - src_min)

def draw_nx_graph_with_both_arrows(key, edges_dict):
    G = nx.DiGraph()

    # Add edges with weights
    for (src, dst), w in edges_dict.items():
        src_clean = src.replace('.csv', '')
        dst_clean = dst.replace('.csv', '')
        G.add_edge(src_clean, dst_clean, weight=w)

    # Calculate node weights (sum of connected edge weights)
    node_weight = {}
    for node in G.nodes():
        in_w = sum(data['weight'] for _, _, data in G.in_edges(node, data=True))
        out_w = sum(data['weight'] for _, _, data in G.out_edges(node, data=True))
        node_weight[node] = in_w + out_w

    weights = list(node_weight.values())
    min_w, max_w = min(weights), max(weights)

    px_min, px_max = 5, 20
    sizes = []
    for w in weights:
        diameter = scale(w, min_w, max_w, px_min, px_max)
        radius = diameter / 2
        area = 3.1415 * (radius ** 2)
        sizes.append(area * 10)

    colors = list(mcolors.TABLEAU_COLORS.values())
    random.shuffle(colors)
    color_map = {node: colors[i % len(colors)] for i, node in enumerate(G.nodes())}
    node_colors = [color_map[node] for node in G.nodes()]

    pos = nx.spring_layout(G, seed=42)

    plt.figure(figsize=(60, 60))

    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_colors, edgecolors='black')

    # Draw edges with offset connectionstyle for parallel edges
    # We'll check if opposite edge exists, then offset lines accordingly
    for u, v, data_edge in G.edges(data=True):
        weight = data_edge['weight']
        # Check if reverse edge exists
        if G.has_edge(v, u):
            # Offset line to left or right for parallel edges
            rad = 0.2 if (u < v) else -0.2
        else:
            rad = 0.0

        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=weight * 0.3,  # thinner edges (scaled down)
            arrowstyle='-|>',
            arrowsize=12,
            connectionstyle=f'arc3,rad={rad}',
            edge_color='gray'
        )

    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

    edge_labels = {(u, v): str(data['weight']) for u, v, data in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10)

    plt.title(f"Graph for {key}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Example: generate both graphs
for k, v in rule_freq.items():
    g = draw_graphviz_scaled_nodes(k, v)
    g.render(filename=f"graph_{k}", cleanup=True)  # Saves to PNG


In [ ]:
for k,v in rule_freq.items():
    print(k,"--",v)

In [ ]:
type(rule_freq)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import random
import textwrap


def scale(value, src_min, src_max, dst_min, dst_max):
    if src_max == src_min:
        return (dst_min + dst_max) / 2
    return dst_min + (value - src_min) * (dst_max - dst_min) / (src_max - src_min)

def wrap_label(label, width=15):
    # Wrap long labels by inserting newlines every 'width' chars
    return '\n'.join(textwrap.wrap(label, width))

# map node colors
all_nodes = set()
for edges_dict in rule_freq.values():
    for src, dst in edges_dict.keys():
        all_nodes.add(src.replace('.csv', ''))
        all_nodes.add(dst.replace('.csv', ''))

colors = list(mcolors.TABLEAU_COLORS.values())
random.shuffle(colors)  # Shuffle once for variety
color_map = {node: colors[i % len(colors)] for i, node in enumerate(sorted(all_nodes))}


def draw_multidigraph_with_arrow_margin(key, edges_dict, color_map):
    G = nx.MultiDiGraph()

    # Add edges
    for (src, dst), w in edges_dict.items():
        src_clean = src.replace('.csv', '')
        dst_clean = dst.replace('.csv', '')
        G.add_edge(src_clean, dst_clean, weight=w)

    # Compute node weights
    node_weight = {}
    for node in G.nodes():
        in_w = sum(data['weight'] for _, _, data in G.in_edges(node, data=True))
        out_w = sum(data['weight'] for _, _, data in G.out_edges(node, data=True))
        node_weight[node] = in_w + out_w

    weights = list(node_weight.values())
    min_w, max_w = min(weights), max(weights)

    px_min, px_max = 5, 20
    sizes = [3.1415 * (scale(w, min_w, max_w, px_min, px_max) / 2) ** 2 * 10 for w in weights]

    node_colors = [color_map[node] for node in G.nodes()]

    # Setup plot
    plt.figure(figsize=(5, 5))
    ax = plt.gca()
    pos = nx.spring_layout(G, seed=42, k=1)

    # Margins
    x_values, y_values = zip(*pos.values())
    x_margin = (max(x_values) - min(x_values)) * 0.15
    y_margin = (max(y_values) - min(y_values)) * 0.15
    ax.set_xlim(min(x_values) - x_margin, max(x_values) + x_margin)
    ax.set_ylim(min(y_values) - y_margin, max(y_values) + y_margin)

    # Draw nodes & labels
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_colors,
                           edgecolors='black', linewidths=1.5)
    wrapped_labels = {node: wrap_label(node, width=15) for node in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=wrapped_labels,
                            font_size=7, font_weight='bold')

    # Edge weights
    all_weights = [d['weight'] for _, _, _, d in G.edges(keys=True, data=True)]
    min_ew, max_ew = min(all_weights), max(all_weights)
    node_size_dict = dict(zip(G.nodes(), sizes))

    for u, v, edge_key, data_edge in G.edges(keys=True, data=True):
        w = data_edge['weight']
        width = scale(w, min_ew, max_ew, 1, 5)
        rad = 0.3 * (edge_key * 2 - 1)
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        dx = x2 - x1
        dy = y2 - y1
        dist = (dx**2 + dy**2) ** 0.5
        if dist == 0:
            continue

        r_source = (node_size_dict[u] / 3.1415) ** 0.5 / 100
        r_target = (node_size_dict[v] / 3.1415) ** 0.5 / 100
        margin_source = r_source + 0.02
        margin_target = r_target + 0.02
        start_x = x1 + dx * margin_source / dist
        start_y = y1 + dy * margin_source / dist
        end_x = x2 - dx * margin_target / dist
        end_y = y2 - dy * margin_target / dist

        arrow = mpatches.FancyArrowPatch(
            (start_x, start_y), (end_x, end_y),
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle='-|>',
            mutation_scale=15,
            linewidth=width,
            color='gray'
        )
        ax.add_patch(arrow)

        # Label closer to target node
        t = 0.75
        base_x = x1 + t * dx
        base_y = y1 + t * dy
        offset_x = -dy / dist * rad * 0.5
        offset_y = dx / dist * rad * 0.5
        label_x = base_x + offset_x
        label_y = base_y + offset_y
        ax.text(label_x, label_y, str(w), fontsize=7,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
                ha='center', va='center')

    plt.title(f"Graph for {key}", fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


for k, v in rule_freq.items():
    draw_multidigraph_with_arrow_margin(k, v, color_map)


# Find context

In [ ]:
results_df.FILE_NAME.tolist()

In [ ]:
results_cand_contexts = {}

for table_path in results_df.FILE_NAME.tolist():
    df = pd.read_csv('../data/events/'+ table_path)
    
    matching_cols = []
    for col in df.columns:
        nunique = df[col].nunique(dropna=False)  # count NaN as unique value
        if 2 <= nunique <= 5:
            col_values = df[col].dropna().astype(str)
            if all(re.fullmatch(r'[A-Za-z\s]+', val) for val in col_values):
                matching_cols.append(col)
    
    results_cand_contexts[table_path] = matching_cols

# Convert results to a DataFrame for easy viewing
matching_cols_df = pd.DataFrame([
    {"TABLE_NAME": table, "COLUMNS": cols} 
    for table, cols in results_cand_contexts.items()
])

print(matching_cols_df)


In [ ]:
matching_cols_df

In [ ]:
results_cand_contexts = {}

for table_path in results_df.FILE_NAME.tolist():
    df = pd.read_csv('../data/events/' + table_path)
    
    matching_cols = []
    for col in df.columns:
        total_rows = len(df)
        nan_ratio = df[col].isna().mean()  
        
        if nan_ratio < 0.3:  # NaN must be less than 30%
            col_values = df[col].dropna().astype(str).str.upper()  # remove NaN and capitalize
            nunique = col_values.nunique()
            
            if 2 <= nunique <= 5:
                # Check all values are letters or spaces
                if True:#all(re.fullmatch(r'[A-Z\s]+', val) for val in col_values):
                    # Check each unique value appears at least 10 times
                    value_counts = col_values.value_counts()
                    if (value_counts >= 10).all():
                        matching_cols.append(col)
    
    results_cand_contexts[table_path] = matching_cols

# Convert results to DataFrame
matching_cols_df = pd.DataFrame([
    {"TABLE_NAME": table, "COLUMNS": cols} 
    for table, cols in results_cand_contexts.items()
])

print(matching_cols_df)

In [ ]:
matching_cols_df

In [ ]:
testn = pd.read_csv(f'../data/events/SCD_TRANSACTION.SCDAT.TRANSSETTLE.csv')

In [ ]:
testn.DSFCTEXT9.value_counts()

In [ ]:
reference_tables

In [ ]:
rule_freq

In [ ]:
def get_potential_contexts(file_path, min_unique=2, max_unique=10):
    """Find columns with 2–10 unique values."""
    data = pd.read_csv(file_path, index_col=0)
    cols = []
    for column in data.columns:
        n_unique = data[column].nunique()
        if min_unique <= n_unique <= max_unique:
            cols.append((file_path, column))
    return cols

def table_counts(seq_dict):
    """Count how many times each table appears in all sequence tuples."""
    t_dict = {}
    for seqs in seq_dict.values():        # each value is a dict of tuples
        for table_pair in seqs.keys():    # table_pair is a tuple like ('file1.csv', 'file2.csv')
            for table in table_pair:      # iterate over both elements in the tuple
                t_dict[table] = t_dict.get(table, 0) + 1
    return t_dict

t_cols = []
table_c = table_counts(rule_freq)  # now using your sequence format

In [ ]:
t_cols

In [ ]:
table_c

In [ ]:
for table in table_c:
    if table_c[table] >= 3:  # only frequent tables
        file_path = os.path.join(fpath, table)
        if os.path.exists(file_path):
            t_cols += get_potential_contexts(file_path)

print(t_cols)

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss

accuracy = {}
log_losses = {}

clf = RandomForestClassifier(n_estimators=500, max_depth=4, random_state=42)
oh_scaler = OneHotEncoder()



# Loop over primary keys
for p_key, seq_dict in rule_freq.items():
    print(f"\n=== Processing primary key: {p_key} ===, seq_dict:{seq_dict}")

    all_data = []
    # Build combined dataset per p_key
    for seq_tuple, count in seq_dict.items():
        for table in seq_tuple:
            table_path = os.path.join(fpath, table)
            if os.path.exists(table_path):
                df = pd.read_csv(table_path)
                if p_key in df.columns:
                    print(f"pkey{p_key} in {table}")
                    df = df.copy()
                    df["sequence_label"] = str(seq_tuple)
                    df["source_table"] = table
                    all_data.append(df)
                else:
                    print(f"Skipping {table} — no {p_key} column")
            else:
                print(f"File not found: {table_path}")

    if not all_data:
        print(f"No data found for {p_key}")
        continue

    full_df = pd.concat(all_data, ignore_index=True).dropna()

    # Process each table separately
    for table in full_df["source_table"].unique():
        sub_df = full_df[full_df["source_table"] == table]
        dlen = len(sub_df)

        if dlen <= 3:  # lower threshold
            print(f"Skipping {table} for {p_key} — only {dlen} rows")
            continue

        try:
            cols = [c for c in sub_df.columns if c not in [p_key, "sequence_label", "source_table"]]
            if len(cols) == 0:
                print(f"Skipping {table} for {p_key} — no features left")
                continue

            X = oh_scaler.fit_transform(sub_df[cols])
            y = sub_df["sequence_label"]

            clf.fit(X, y)
            yp = clf.predict(X)
            ypred = clf.predict_proba(X)

            log_losses[(p_key, table, dlen)] = log_loss(y, ypred)
            accuracy[(p_key, table, dlen)] = np.mean(y == yp)

            # Feature importances
            feature_names = oh_scaler.get_feature_names_out(cols)
            feature_importances = clf.feature_importances_

            forest_importances = pd.DataFrame({
                "feature": feature_names,
                "importance": feature_importances
            })

            # Aggregate by original column name (remove one-hot suffix)
            forest_importances['Column Name'] = forest_importances['feature'].apply(lambda x: '_'.join(x.split('_')[:-1]))
            df_ss = forest_importances.groupby('Column Name')['importance'].sum().sort_values(ascending=False)

            print(table[:-4], dlen)
            display(df_ss)

        except Exception as e:
            print(f"Skipping {table} for {p_key} due to error: {e}")

print("\n=== Done ===")


In [ ]:
# View all accuracy results
for key, acc in accuracy.items():
    p_key, table, rows = key
    print(f"Primary key: {p_key}, Table: {table}, Rows: {rows}, Accuracy: {acc:.2f}")


In [ ]:
accuracy

In [ ]:
display(df_ss)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
accuracy = {}

clf = RandomForestClassifier(n_estimators=500, max_depth=4, random_state=42)
oh_scaler = OneHotEncoder()

plt.figure(figsize=(16, 16))

for p_key, seq_dict in rule_freq.items():
    print(f"\n=== Processing primary key: {p_key} ===")
    
    # Build dataset for this p_key
    all_data = []
    for seq_tuple, count in seq_dict.items():
        for table in seq_tuple:
            table_path = os.path.join(fpath, table)
            if os.path.exists(table_path):
                df = pd.read_csv(table_path)
                if p_key in df.columns:
                    df = df.copy()
                    df["sequence_label"] = str(seq_tuple)
                    df["source_table"] = table
                    all_data.append(df)
                else:
                    print(f"Skipping {table} — no {p_key} column")
            else:
                print(f"File not found: {table_path}")

    if not all_data:
        print(f"No data found for {p_key}")
        continue

    full_df = pd.concat(all_data, ignore_index=True).dropna()
    
    # Loop over tables
    for table in full_df["source_table"].unique():
        tname = table.split('/')[-1]
        sub_df = full_df[full_df["source_table"] == table]
        dlen = len(sub_df)
        
        if dlen <= 10:
            continue
        
        try:
            cols = [c for c in sub_df.columns if c not in [p_key, "sequence_label", "source_table"]]
            X = oh_scaler.fit_transform(sub_df[cols])
            y = sub_df["sequence_label"]

            clf.fit(X, y)
            yp = clf.predict(X)
            accuracy[(p_key, table, dlen)] = np.mean(y == yp)

            # Feature importances
            feature_names = oh_scaler.get_feature_names_out(cols)
            feature_importances = clf.feature_importances_
            
            forest_importances = pd.DataFrame({
                "feature": feature_names,
                "importance": feature_importances
            })

            average_importance = np.median(forest_importances["importance"])
            df_ss = forest_importances[forest_importances["importance"] > average_importance]

            print(tname[:-4], dlen)
            display(df_ss)

            sns.catplot(data=df_ss,
                        y="importance",
                        x="feature",
                        kind="bar",
                        height=6,
                        aspect=2)
            plt.xticks(rotation=90)
            plt.title(f'{tname[:-4]}')

        except Exception as e:
            print(f"Skipping {table} for {p_key} due to: {e}")


In [ ]:
# View all accuracy results
for key, acc in accuracy.items():
    p_key, table, rows = key
    print(f"Primary key: {p_key}, Table: {table}, Rows: {rows}, Accuracy: {acc:.2f}")


In [ ]:
def expand_pairs_all_keys(rules_dict):
    results = {}
    
    for key, sequences in rules_dict.items():
        pair_counts = defaultdict(int)
        
        # Step 1: Count all pairs from tableA
        for seq, count in sequences.items():
            for i in range(len(seq) - 1):
                pair = (seq[i], seq[i+1])
                pair_counts[pair] += count
        
        # Step 2: Create DataFrame template
        df = pd.DataFrame(
            [{'TableA': a, 'TableB': b, 'count': c} for (a, b), c in pair_counts.items()]
        )
        
        # Step 3: Calculate total flows from each TableA
        totals = df.groupby('TableA')['count'].transform('sum')
        
        # Step 4: Calculate M as a percentage M = C_(Ti,Tj)/Sum(C_(Ti,Tj)...)
        df['M'] = (df['count'] / totals) * 100
        
        # Step 5: Sort & format
        df = df.sort_values(by=['TableA', 'TableB']).reset_index(drop=True)
        results[key] = df
    
    return results

In [ ]:
# Run
dfs = expand_pairs_all_keys(rule_freq)

# Example: show all
for key, df in dfs.items():
    print(f"\n=== {key} ===")
    print(df)

In [ ]:
dfs['TRANSIK']

In [ ]:
threshold_theta = # user defined minimum difference in transition probabilities

### random sampling of data based on Pkey

In [ ]:
def generate_new_data(count_dict,
                      primary_key   = pkey,
                      n_iter        = 1):
    
    ### This function takes the pk/id count dictionary, and randomly samples the key for 2000 observations
    
    keys_list                     = []
    sequences                     = []
    
    keys_sampled                  = random.sample(count_dict_ss.keys(), n_iter)
    
    for key in keys_sampled:
        keys_list.append(key)
        sequences.append(count_dict[key])
        
    return keys_list, sequences  # Return just the table names in sequence

In [ ]:
num_iter = 20# modifired later

ids, seqs = generate_new_data(count_dict_ss, n_iter=num_iter)
df        = pd.DataFrame({'Sequence': seqs}, index=ids)
df.to_csv('output_flow/scd_sequences.csv')

# Primary Key Clustering and Context Discovery 

In [ ]:
random_state  = 0
threshold     = 0.05

In [ ]:
def read_data(d_fpath, p_key):
    
    ### Read data and convert sequences from strings to lists
    ### The ID column is the primary key, while the "Sequence" column is a list of tables traversed by the ID
    
    data = pd.read_csv(d_fpath, index_col=0)
    data.reset_index(names=p_key, inplace=True)
    
    data['Sequence'] = data['Sequence'].apply(lambda x: ast.literal_eval(x))
    return data

def dt_column(cols): 
    
    dt_count = {}
    
    for col in cols:
        dt_count[col] = dt_count.get(col, 0) + 1
    
    return {k: v for k, v in sorted(dt_count.items(), key=lambda item: item[1], reverse=True)[:5]}


In [ ]:
# Read data and datetime columns data
data   = read_data(dpath, p_key)
ids    = data[p_key].tolist()
df_dt  = pd.read_csv(dt_path)

# Observe most common datetime columns
columns = [col for cols in df_dt['DATETIME_COL_SELECTED'].dropna().apply(lambda x: x.split(', ')).tolist() for col in cols]
display(dt_column(columns))



In [ ]:
column_counts = Counter(columns)
sorted_columns = sorted(column_counts.items(), key=lambda x: x[1], reverse=True)

# Print in descending order
for col, count in sorted_columns:
    print(f"{col}: {count}")

# Save the most frequent column
fre_column = sorted_columns[0][0]
print(f"\nMost frequent column: {fre_column}")

In [ ]:
fre_column

In [ ]:
# Drop data tables that do not contain the most common datetime columns
dcolumns  = [fre_column]# the most frequent column
df_dt_c   = df_dt.dropna()
df_dt_ss  = df_dt_c[df_dt_c['DATETIME_COL_SELECTED'].apply(lambda x: len([c for c in x.split(', ') if c in columns]) > 0)]
tables    = set([t[:-4] for t in df_dt_ss['FILE_NAME']])

In [ ]:
# dataset_folder = "events"
# folder_path = f'../data/{dataset_folder}'
# print('running...')
# 
# similarity_df = compute_similarity_with_ks(folder_path)
# print(similarity_df)
# 
# similarity_df.to_csv(f'../data/reference/{dataset_folder}_similarity_output.csv', index=False)

In [ ]:
dt_columns          = df_dt_c.apply(lambda x: [(x['FILE_NAME'], datetime_col)  
                                               for datetime_col in x['DATETIME_COL_SELECTED'].split(', ')], axis=1).tolist()
dt_columns_unzipped = [(df[:-4], dt_col) for dt_cols in dt_columns for df, dt_col in dt_cols]

In [ ]:
# candidates for datetime columns
dt_columns


In [ ]:
dt_columns_unzipped

## generate sequences
We create a binary dataframe with the 1000 randomly sampled ids as its index and the list of all data tables. Then, we create 2 additional dataframes:

1. df_recurrence: a data table that counts the number of occurrences a particular table appears for an id
2. df_sequence: a data table that maps the first instance of the id appearing in a particular table.

In [ ]:
def generate_ids(fpath, ids, p_key):
    
    # Create a dataframe with binary values (whether table exists for id) with ids as index, and table names as columns.
    
    files = [file[:-4] for file in os.listdir(output_path) if file.endswith('.csv')]
    print(files)
    df    = pd.DataFrame(index=ids, columns=files)
    
    for file in files:
        try: 
            ids = pd.read_csv(os.path.join(fpath, file+'.csv'))[p_key].tolist()

            for idx in ids:
                df.at[idx, file] = 1
        
        except:
            pass

    return df
    
data = generate_ids(output_path, ids, p_key) 

In [ ]:
def clean_sequences(data, dt_tuples, fpath, pkey=p_key):
    
    ### For each sequence, create 2 dataframes, df_rec and df_seq
    
    ### df_rec (or df_recurrence) is a dataframe with rows as observations (or IDs), and columns as table names
    ### the values each cell in df_rec can take will either be NA (or 0), 1, 2, 3... 
    ### the cell value depends on the number of times a particular table appears for an observation of interest
    
    ### df_seq (or df_sequence) is a dataframe with rows as observations (or IDs), and columns as table names
    ### the values within df_seq is a timestamp
    ### this depends on the earliest timestamp an observation is observed in the particular table (or min)
    
    df_rec  = pd.DataFrame(index=data.index, columns=data.columns)
    df_seq  = pd.DataFrame(index=data.index, columns=data.columns)
    
    for table, dt_column in dt_tuples:
        
        if table in df_seq.columns:
            table_name = table + '.csv'
            table_path = os.path.join(fpath, table_name)

            df_columns = [p_key, dt_column]
            
            try: 
                data       = pd.read_csv(table_path)
                data_ss    = data[df_columns].set_index(p_key)

                counter = 0
                for idx in df_seq.index:
                    if idx in data_ss.index:         
                        df_seq.at[idx, table] = np.min([pd.to_datetime(data_ss.loc[idx][dt_column])])
                        df_rec.at[idx, table] = len([data_ss.loc[idx][dt_column]])
            
            except:
                pass

    
    return df_rec, df_seq
    
df_recurrence, df_sequence = clean_sequences(data, dt_tuples, output_path, pkey=p_key)

#### data preprocessing
We conduct the following data pre-processing steps:

Remove tables that display little variation in insertion times (this suggests bulk uploading and table may not be informative of the underlying process that an id is tied to).
Remove duplicated ids (we are only interested in the first instance in which the id appears, but duplicated ids may suggest non-STP)